# CO2-Aware Workload Migration: Where Should SURF Run Its Workload?

*Use case 3 of 3, about 10 minutes.*

We envision digital twins becoming primary instruments in datacenter operations, continuously and autonomously helping
with major operational decisions and with adapting ICT infrastructure, live, with a human-in-the-loop.
In this use case, OpenDT acts as an advisor-on-the-side for an operational technique: CO2-aware workload migration.

The CO2 emissions of a datacenter depend on its energy and on the carbon intensity of the electricity grid
[gCO2/kWh] at the time of use, which differs from country to country and from hour to hour.
We ask OpenDT: if SURF ran the same workload, on the same hardware, in another country, how much CO2 would it emit?
We compare four grids: the Netherlands (NL), Germany (DE), Romania (RO), and Poland (PL).

In this notebook, we (1) compare the carbon intensity of the four grids, (2) simulate the SURF workload on each grid,
and (3) discuss the advice of OpenDT with the human-in-the-loop.

Run the cells in order (Shift+Enter). When short on time, skip the parts marked *optional*.

In [ ]:
import pandas as pd
from opendt_tutorial import *

check_setup()

## 1. Carbon Intensity of Four Grids

The carbon intensity data comes from Electricity Maps (October 2022, hourly, ODbL license; see
`carbon_traces/LICENSE.md`).

*Question:* which of the four grids do we expect to be the greenest during the SURF week? Write down a guess, then run
the next cell.

In [ ]:
intensity = pd.DataFrame({zone: carbon_intensity(zone)[TRACE_START:TRACE_END] for zone in CARBON_ZONES})

ax = intensity.plot(figsize=(10, 4))
ax.set(xlabel="Time [UTC]", ylabel="Carbon Intensity [gCO2/kWh]", ylim=(0, None))
ax.yaxis.set_major_formatter("{x:,.0f}")  # 1,000
legend_above(ax)
intensity.mean().round().rename("Average [gCO2/kWh]").to_frame().T

*Question:* is the same grid the greenest at every hour?

<details><summary>Answer</summary>

No. RO has the lowest average (343 gCO2/kWh), but NL is greener in 57 of the 169 hours; the carbon intensity
of NL changes more over the week (from 123 to 528 gCO2/kWh).
</details>

## 2. One Simulation per Grid

OpenDT simulates the full week (6-13 October 2022) once per grid; only the carbon intensity trace of the topology
changes (`powerSource.carbonTracePath`).

*Question:* what do we expect to change between the four simulations, and what to stay the same? Write down a guess,
then run the next cell.

In [ ]:
runs = simulate_many("carbon", CARBON_ZONES, until=TRACE_END)  # one simulation of the full week per grid

table = pd.DataFrame.from_dict({zone: run.summary() for zone, run in runs.items()}, orient="index")
table[["energy_kwh", "co2_kg"]].round()

<details><summary>Answer</summary>

The energy is the same in the four simulations (3,424 kWh): the workload and the hardware do not change.
The CO2 emissions change with the carbon intensity of the grid, from 1,172 kg (RO) to 2,959 kg (PL).
</details>

OpenDT, as an advisor-on-the-side, turns these simulations into advice:

In [ ]:
ax = table["co2_kg"].sort_values(ascending=False).plot.barh(figsize=(8, 2.5), color="#009E73")
ax.set(xlabel="CO2 Emissions [kg]", ylabel="")
ax.xaxis.set_major_formatter("{x:,.0f}")

best = table["co2_kg"].idxmin()
saving = table.loc["NL", "co2_kg"] - table.loc[best, "co2_kg"]
print(f"Advice: run the SURF workload in {best}, with {saving:,.0f} kg CO2 less per week than in NL "
      f"({saving / table.loc['NL', 'co2_kg']:.0%}).")

## 3. The Human-in-the-Loop

*Question:* should SURF follow this advice? What else should the human-in-the-loop consider before migrating the
workload?

<details><summary>Answer</summary>

The saving is small (6% vs. NL), and NL is greener than RO in a third of the hours.
Before migrating, the human-in-the-loop considers, e.g., (1) the energy and time to move data and jobs over the
network, (2) the latency for the users, (3) the rules on where data may be stored, (4) the free capacity and the cost
of the remote datacenter, and (5) the CO2 emitted to build new hardware.
In OpenDT, the feedback to the physical ICT is SLO-aware and human-approved.
</details>

*Exercise 1:* if the workload could migrate every hour to the greenest of the four grids, how much CO2 would SURF emit?
Run the next cell.

In [ ]:
co2_per_hour = pd.DataFrame({zone: run.power["carbon_emission"].resample("1h").sum() / 1000 for zone, run in runs.items()})
greenest = co2_per_hour.idxmin(axis=1)  # the greenest grid in each hour
switches = (greenest != greenest.shift())[1:].sum()  # migrations from one hour to the next
print(f"Greenest grid every hour: {co2_per_hour.min(axis=1).sum():,.0f} kg CO2, with {switches} migrations")

<details><summary>Answer</summary>

1,065 kg, 15% less than in NL and 9% less than in RO. However, the workload migrates 11 times in one week,
between NL and RO, and each migration costs energy and time.
</details>

*Exercise 2 (optional):* combine with notebook 1: simulate the week in RO with 1/2 of the hosts, and compare with 277
hosts in NL.

In [ ]:
half = simulate(until=TRACE_END, hosts=138, carbon="RO", label="RO, 138 hosts")
pd.DataFrame({"NL, 277 hosts": runs["NL"].summary(), "RO, 138 hosts": half.summary()}).round(1)

<details><summary>Answer</summary>

In RO with 138 hosts, SURF emits 971 kg CO2, 23% less than with 277 hosts in NL.
However, 2,631 jobs wait for a free host, and the average waiting time per job is 78 minutes.
</details>

## Summary

In this notebook, we (1) compare the carbon intensity of four grids, (2) use OpenDT as an advisor-on-the-side for
CO2-aware workload migration, and (3) discuss the advice with the human-in-the-loop.

*Next (optional):* run the live digital twin yourself, in [4.live_twin.ipynb](4.live_twin.ipynb).

*Learn more:* the [topology configuration](https://atlarge-research.github.io/opendt/docs/configuration/topology) (`carbonTracePath`), the
[OpenDT paper](https://atlarge-research.com/pdfs/2026-icpe-opendt.pdf), and
[Electricity Maps](https://www.electricitymaps.com).